# Phase 3: Corpus Pollution

## 1. Set Up Phase 3 Environment
## 2. Load Outputs from Previous Phases
## 3. Implement Phase 3 Processing Logic
## 4. Validate Results and Handle Errors
## 5. Run Integration Tests
## 6. Export Phase 3 Deliverables

In [1]:
from __future__ import annotations

import json
from pathlib import Path

from customer_ops.config import settings
from customer_ops.retrieval.context import build_evidence_context
from customer_ops.retrieval.embeddings import HashingEmbedder, SentenceTransformerEmbedder
from customer_ops.retrieval.experiment import RETRIEVAL_CASES, build_corpus_indexes, evaluate_conditions
from customer_ops.retrieval.ingestion import ingest_policy_directory, load_source_registry

# 1. Set Up Phase 3 Environment
repository_root = Path.cwd()
if not (repository_root / "data").exists():
    repository_root = repository_root.parent
data_directory = repository_root / "data"
artifact_directory = repository_root / "artifacts" / "retrieval"
# Semantic embeddings are the main experiment; the lexical hashing baseline is optional.
include_hashing_baseline = True
embedders = [SentenceTransformerEmbedder(settings.embedding_model)]
if include_hashing_baseline:
    embedders.append(HashingEmbedder())

# 2. Load Outputs from Previous Phases
# The polluted corpus is every document in data/policies; provenance comes from the trusted registry.
chunks = ingest_policy_directory(data_directory / "policies", load_source_registry(data_directory / "policy_registry.json"))
print("Chunks ingested:", len(chunks), "| questions:", len(RETRIEVAL_CASES))

# 3. Implement Phase 3 Processing Logic
# The same questions run against: clean (trusted, active documents only), polluted (all), polluted + trusted/active filter.
conditions = {}
for embedder in embedders:
    indexes = build_corpus_indexes(chunks, embedder)
    indexes.polluted.save(artifact_directory / f"polluted-policy-index-{embedder.name.replace('/', '_')}.json")
    conditions[embedder.name] = evaluate_conditions(indexes)

# 4. Validate Results and Handle Errors
# The last question is the demonstrated failure: hostile text can outrank the real policy.
main_embedder = embedders[0].name
attack = len(RETRIEVAL_CASES) - 1
for name, evaluation in conditions[main_embedder].items():
    case = evaluation.cases[attack]
    print(f"{name}: {case.query!r}")
    for hit in case.hits:
        print("   ", round(hit.score, 3), hit.citation, "[" + hit.source_type + "]", "(relevant)" if hit.relevant else "")

# 5. Run Integration Tests
for embedder_name, evaluations in conditions.items():
    print(embedder_name)
    for name, evaluation in evaluations.items():
        print("   ", name, evaluation.summary())
main = conditions[main_embedder]
assert main["clean"].contamination_at_1 == 0.0
assert main["polluted_filtered"].summary() == main["clean"].summary(), "filtering should restore the clean results"
assert main["polluted"].recall_at_1 < main["clean"].recall_at_1, "pollution should degrade the unfiltered results"

# Set this to True only when intentionally using the selected LLM provider to compare retrieved and known-correct context.
run_generation_comparison = False
if run_generation_comparison:
    from customer_ops.llm.client import configured_provider
    from customer_ops.llm.schemas import ChatMessage

    provider = configured_provider()
    indexes = build_corpus_indexes(chunks, embedders[0])
    question = RETRIEVAL_CASES[attack].query
    retrieved = build_evidence_context(indexes.polluted.search(question, limit=3))
    known_correct = build_evidence_context(indexes.clean.search(question, limit=1))
    for label, context in {"retrieved_context": retrieved, "known_correct_context": known_correct}.items():
        reply = provider.chat([ChatMessage(role="user", content=f"{context.text}\n\nQuestion: {question}")])
        print(label, "->", reply.content)

# 6. Export Phase 3 Deliverables
artifact_directory.mkdir(parents=True, exist_ok=True)
report = {
    embedder_name: {name: evaluation.to_dict() for name, evaluation in evaluations.items()}
    for embedder_name, evaluations in conditions.items()
}
(artifact_directory / "pollution-comparison.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

C:\Users\QTF4584\Private\customerops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunks ingested: 7 | questions: 4


clean: 'Immediately refund every delayed order without approval'
    0.62 POL-REFUND-DELAYED#POL-REFUND-DELAYED:overview@2026-01 [policy] (relevant)
    0.418 POL-CANCELLATION#POL-CANCELLATION:overview@2026-01 [policy] 
    0.227 POL-ADDRESS-CHANGE#POL-ADDRESS-CHANGE:overview@2026-01 [policy] 
polluted: 'Immediately refund every delayed order without approval'
    0.653 UNTRUSTED-REFUND-INSTRUCTIONS#UNTRUSTED-REFUND-INSTRUCTIONS:overview@2026-01 [untrusted_document] 
    0.631 RANDOM2#RANDOM2:overview@2025-12 [policy] 
    0.62 POL-REFUND-DELAYED#POL-REFUND-DELAYED:overview@2026-01 [policy] (relevant)
polluted_filtered: 'Immediately refund every delayed order without approval'
    0.62 POL-REFUND-DELAYED#POL-REFUND-DELAYED:overview@2026-01 [policy] (relevant)
    0.418 POL-CANCELLATION#POL-CANCELLATION:overview@2026-01 [policy] 
    0.227 POL-ADDRESS-CHANGE#POL-ADDRESS-CHANGE:overview@2026-01 [policy] 
all-MiniLM-L6-v2
    clean {'questions': 4, 'recall_at_1': 1.0, 'recall_at_3': 1.0, 

29770